# Download Dataset

In [ ]:
!mkdir -p ~/.kaggle
!cp /your Kaggle json path ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download monishshrivastava1/llvip-dataset -p /content

Extract Dataset

In [ ]:
import zipfile
import os

zip_path = '/content/llvip-dataset.zip'
extract_path = '/content/ImageFusionProject'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Çıkarma tamamlandı.")

Getting Data

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

class FusionDataset(Dataset):
    def __init__(self, vis_dir, ir_dir, transform=None):
        self.vis_dir = vis_dir
        self.ir_dir = ir_dir
        self.transform = transform


        self.image_filenames = sorted(os.listdir(vis_dir))

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]

        vis_path = os.path.join(self.vis_dir, img_name)
        ir_path = os.path.join(self.ir_dir, img_name)


        vis_image = Image.open(vis_path).convert('RGB')
        ir_image = Image.open(ir_path).convert('L')

        if self.transform:
            vis_image = self.transform(vis_image)
            ir_image = self.transform(ir_image)

        return vis_image, ir_image


data_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])
VIS_TRAIN_DIR = '/content/ImageFusionProject/LLVIP/visible/train'
IR_TRAIN_DIR = '/content/ImageFusionProject/LLVIP/infrared/train'



train_dataset = FusionDataset(vis_dir=VIS_TRAIN_DIR, ir_dir=IR_TRAIN_DIR, transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)

print(f"Toplam eğitim verisi çifti: {len(train_dataset)}")

# Training Setup

Color Setup

In [ ]:
def rgb_to_ycbcr(image):

    r, g, b = image[:, 0:1, :, :], image[:, 1:2, :, :], image[:, 2:3, :, :]
    y = 0.299 * r + 0.587 * g + 0.114 * b
    cb = -0.1687 * r - 0.3313 * g + 0.5 * b + 0.5
    cr = 0.5 * r - 0.4187 * g - 0.0813 * b + 0.5
    return y, cb, cr
def ycbcr_to_rgb(y, cb, cr):
    cb = cb - 0.5
    cr = cr - 0.5
    r = y + 1.402 * cr
    g = y - 0.34414 * cb - 0.71414 * cr
    b = y + 1.772 * cb
    return torch.cat([r, g, b], dim=1).clamp(0, 1)

Illumunation Model Load (check path)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, padding_mode='reflect'),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, padding_mode='reflect'),
            nn.LeakyReLU(0.2, inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class SharedEncoder(nn.Module):
    def __init__(self):
        super(SharedEncoder, self).__init__()
        self.enc1 = ConvBlock(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(128, 256)

    def forward(self, x):
        f1 = self.enc1(x)
        f2 = self.enc2(self.pool1(f1))
        f3 = self.enc3(self.pool2(f2))
        bottle = self.bottleneck(self.pool3(f3))
        return bottle, f3, f2, f1

class DecoderBranch(nn.Module):
    def __init__(self, is_illumination=False):
        super(DecoderBranch, self).__init__()
        self.is_illumination = is_illumination

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128)
        self.out_quarter = nn.Conv2d(128, 3, kernel_size=1)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64)
        self.out_half = nn.Conv2d(64, 3, kernel_size=1)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32)
        self.out_full = nn.Conv2d(32, 3, kernel_size=1)

    def forward(self, bottle, f3, f2, f1):

        x = self.up3(bottle)
        x = torch.cat([x, f3], dim=1)
        x = self.dec3(x)
        out_1_4 = self.out_quarter(x)


        x = self.up2(x)
        x = torch.cat([x, f2], dim=1)
        x = self.dec2(x)
        out_1_2 = self.out_half(x)


        x = self.up1(x)
        x = torch.cat([x, f1], dim=1)
        x = self.dec1(x)
        out_1_1 = self.out_full(x)


        if self.is_illumination:
            out_1_4 = torch.sigmoid(out_1_4)
            out_1_2 = torch.sigmoid(out_1_2)
            out_1_1 = torch.sigmoid(out_1_1)

        return out_1_4, out_1_2, out_1_1


class DualBranchDecompNet(nn.Module):
    def __init__(self):
        super(DualBranchDecompNet, self).__init__()
        self.encoder = SharedEncoder()


        self.illumination_branch = DecoderBranch(is_illumination=True)
        self.residual_branch = DecoderBranch(is_illumination=False)

    def forward(self, x):

        bottle, f3, f2, f1 = self.encoder(x)

        ill_1_4, ill_1_2, ill_full = self.illumination_branch(bottle, f3, f2, f1)
        res_1_4, res_1_2, res_full = self.residual_branch(bottle, f3, f2, f1)

        enhanced_1_4 = ill_1_4 + res_1_4
        enhanced_1_2 = ill_1_2 + res_1_2
        enhanced_full = ill_full + res_full

        return {
            'enhanced': [enhanced_1_4, enhanced_1_2, enhanced_full],
            'illumination': [ill_1_4, ill_1_2, ill_full],
            'residual': [res_1_4, res_1_2, res_full]
        }

if __name__ == '__main__':
    dummy_input = torch.randn(8, 3, 256, 256)

    model = DualBranchDecompNet()

    outputs = model(dummy_input)

    print("Multi-Scale Enhanced Outputs Shapes:")
    print("1/4 Ölçek:", outputs['enhanced'][0].shape)
    print("1/2 Ölçek:", outputs['enhanced'][1].shape)
    print("Tam Ölçek:", outputs['enhanced'][2].shape)
    print("\nModel başarıyla çalışıyor! Parametre sayısı:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Load Enhancement Model (Check path)

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

enhancement_module = DualBranchDecompNet().to(device)

pth_path = '/content/checkpoints/illum/epoch_100_spa_0.05_exp_1.0.pth'

enhancement_module.load_state_dict(torch.load(pth_path, map_location=device))

for param in enhancement_module.parameters():
    param.requires_grad = False

enhancement_module.eval()

print("Weights Loaded")

Feature Encoder

In [ ]:
class FeatureEncoder(nn.Module):
    def __init__(self, in_channels=1):
        super(FeatureEncoder, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.relu1 = nn.LeakyReLU(0.2, inplace=True)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.relu2 = nn.LeakyReLU(0.2, inplace=True)
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.relu3 = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.relu3(self.conv3(x))
        return x

Cross Attention

In [ ]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, feature_dim):
        super(CrossAttentionFusion, self).__init__()
        self.query_conv = nn.Conv2d(feature_dim, feature_dim // 8, kernel_size=1)
        self.key_conv = nn.Conv2d(feature_dim, feature_dim // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(feature_dim, feature_dim, kernel_size=1)

        self.gamma = nn.Parameter(torch.ones(1))

    def forward(self, ir_features, vis_features):
        batch_size, C, H, W = ir_features.size()

        proj_query = self.query_conv(ir_features).view(batch_size, -1, H * W).permute(0, 2, 1)
        proj_key = self.key_conv(vis_features).view(batch_size, -1, H * W)
        proj_value = self.value_conv(vis_features).view(batch_size, -1, H * W)

        energy = torch.bmm(proj_query, proj_key)
        attention = F.softmax(energy, dim=-1)

        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        out = out.view(batch_size, C, H, W)

        out = self.gamma * out + ir_features
        return out

Decoder

In [ ]:
class FusionDecoder(nn.Module):
    def __init__(self, in_features=256, out_channels=1):
        super(FusionDecoder, self).__init__()
        self.upconv1 = nn.ConvTranspose2d(in_features, 128, kernel_size=4, stride=2, padding=1)
        self.relu1 = nn.LeakyReLU(0.2, inplace=True)
        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.relu2 = nn.LeakyReLU(0.2, inplace=True)
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.relu1(self.upconv1(x))
        x = self.relu2(self.upconv2(x))
        x = torch.sigmoid(self.final_conv(x))
        return x

Master Class

In [ ]:
class EndToEndFusionNetwork(nn.Module):
    def __init__(self, enhancement_model):
        super(EndToEndFusionNetwork, self).__init__()
        self.enhancement_model = enhancement_model

        self.ir_encoder = FeatureEncoder(in_channels=1)
        self.vis_encoder = FeatureEncoder(in_channels=3)
        self.cross_attention = CrossAttentionFusion(feature_dim=256)
        self.decoder = FusionDecoder(in_features=256, out_channels=1)

    def forward(self, low_light_vis, ir_img):
        with torch.no_grad():
            enhancement_outputs = self.enhancement_model(low_light_vis)
            enhanced_vis_rgb = enhancement_outputs['enhanced'][2]

        ir_features = self.ir_encoder(ir_img)
        vis_features = self.vis_encoder(enhanced_vis_rgb)

        fused_features = self.cross_attention(ir_features, vis_features)
        mask = self.decoder(fused_features)

        y, cb, cr = rgb_to_ycbcr(enhanced_vis_rgb)

        fused_y = (mask * ir_img) + ((1 - mask) * y)

        fused_rgb = ycbcr_to_rgb(fused_y, cb, cr)

        return fused_rgb, mask, enhanced_vis_rgb

Loss Function

In [ ]:
!pip install pytorch-msssim

In [ ]:
import torch.nn as nn
from pytorch_msssim import ssim

class FusionLoss(nn.Module):
    def __init__(self, lambda_pixel=1.0, lambda_ssim=10.0):
        super(FusionLoss, self).__init__()
        self.lambda_pixel = lambda_pixel
        self.lambda_ssim = lambda_ssim
        self.l1_loss = nn.L1Loss()

    def forward(self, fused_rgb, ir_img, vis_rgb):
        ir_rgb = ir_img.repeat(1, 3, 1, 1)

        target = torch.max(ir_rgb, vis_rgb)
        loss_pixel = self.l1_loss(fused_rgb, target)

        fused_gray = fused_rgb.mean(dim=1, keepdim=True)
        vis_gray = vis_rgb.mean(dim=1, keepdim=True)

        loss_ssim_ir = 1 - ssim(fused_gray, ir_img, data_range=1.0, size_average=True)
        loss_ssim_vis = 1 - ssim(fused_gray, vis_gray, data_range=1.0, size_average=True)

        loss_ssim = loss_ssim_ir + loss_ssim_vis
        total_loss = self.lambda_pixel * loss_pixel + self.lambda_ssim * loss_ssim

        return total_loss, loss_pixel, loss_ssim

Training (Check Path) Code optimized for partial training. You can continue later with your lates weight.

In [ ]:
import os
import torch.optim as optim
from tqdm import tqdm

save_dir = '/content/checkpoints/colorcheckpoint'
os.makedirs(save_dir, exist_ok=True)

num_epochs = 105
learning_rate = 1e-4

optimizer = optim.Adam(filter(lambda p: p.requires_grad, full_model.parameters()), lr=learning_rate)
criterion = FusionLoss(lambda_pixel=1.0, lambda_ssim=10.0)


resume_checkpoint_path = os.path.join(save_dir, 'fusion_modelcolorv3_epoch_70.pth')

start_epoch = 0

if os.path.exists(resume_checkpoint_path):
    print(f"Kayıtlı model bulundu: {resume_checkpoint_path}")
    checkpoint = torch.load(resume_checkpoint_path, map_location=device)

    full_model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']

    print(f"Eğitim {start_epoch + 1}. epoch'tan itibaren kaldığı yerden devam edecek!")
else:
    print("Kayıtlı model bulunamadı. Eğitim 1. epoch'tan sıfırdan başlıyor.")
print("Eğitim Başlıyor...")

for epoch in range(start_epoch, num_epochs):
    full_model.train()

    epoch_loss = 0.0
    epoch_pixel_loss = 0.0
    epoch_ssim_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, (vis_img, ir_img) in enumerate(progress_bar):
        vis_img = vis_img.to(device)
        ir_img = ir_img.to(device)

        optimizer.zero_grad()

        fused_rgb, mask, enhanced_vis = full_model(vis_img, ir_img)

        loss, l_pixel, l_ssim = criterion(fused_rgb, ir_img, enhanced_vis)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_pixel_loss += l_pixel.item()
        epoch_ssim_loss += l_ssim.item()

        progress_bar.set_postfix({'Total Loss': loss.item(), 'SSIM Loss': l_ssim.item()})

    avg_loss = epoch_loss / len(train_loader)
    print(f"\n--- Epoch {epoch+1} Özeti ---")
    print(f"Ortalama Toplam Kayıp: {avg_loss:.4f}")

    checkpoint_path = os.path.join(save_dir, f'fusion_modelcolorv3_epoch_{epoch+1}.pth')

    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': full_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
    }, checkpoint_path)

    print(f"[!] Model ve optimizatör durumu yedeklendi: Epoch {epoch+1}\n")

# Test

After Training Run this. If you don't run train do the other code block.

In [ ]:
import matplotlib.pyplot as plt
import torch

full_model.eval()

vis_data, ir_data = train_loader.dataset[2500]

vis_batch = vis_data.unsqueeze(0).to(device)
ir_batch = ir_data.unsqueeze(0).to(device)

with torch.no_grad():
    fused_rgb, mask, enhanced_vis = full_model(vis_batch, ir_batch)

def tensor_to_img(tensor, is_grayscale=False):
    img = tensor[0].cpu().detach()
    if is_grayscale:
        img = img.squeeze(0).numpy()
    else:
        img = img.permute(1, 2, 0).numpy()
    return img.clip(0, 1)

img_vis = tensor_to_img(vis_batch, is_grayscale=False)
img_enhanced = tensor_to_img(enhanced_vis, is_grayscale=False)
img_ir = tensor_to_img(ir_batch, is_grayscale=True)

img_fused = tensor_to_img(fused_rgb, is_grayscale=False)

from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(img_vis)
axes[0, 0].set_title('1. Low Light VIS')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_enhanced)
axes[0, 1].set_title('2. Enhanced VIS')
axes[0, 1].axis('off')

axes[1, 0].imshow(img_ir, cmap='gray')
axes[1, 0].set_title('3. Infrared (IR)')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_fused)
axes[1, 1].set_title('4. Fused')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

Test with selected model (Control Path)

In [ ]:
import matplotlib.pyplot as plt
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

full_model1 = EndToEndFusionNetwork(enhancement_module).to(device)
model_path = "/content/checkpoints/colorcheckpoint/fusion_modelcolorv3_epoch_105.pth"
checkpoint = torch.load(model_path, map_location=device)

try:
    full_model1.load_state_dict(checkpoint)
    print("✅ Ağırlıklar modele doğrudan başarıyla yüklendi!")
except Exception:
    full_model1.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Ağırlıklar sözlük (dictionary) üzerinden başarıyla yüklendi!")

full_model1.eval()

vis_data, ir_data = train_loader.dataset[2500]

vis_batch = vis_data.unsqueeze(0).to(device)
ir_batch = ir_data.unsqueeze(0).to(device)

with torch.no_grad():
    fused_rgb, mask, enhanced_vis = full_model1(vis_batch, ir_batch)

def tensor_to_img(tensor, is_grayscale=False):
    img = tensor[0].cpu().detach()
    if is_grayscale:
        img = img.squeeze(0).numpy()
    else:
        img = img.permute(1, 2, 0).numpy()
    return img.clip(0, 1)

img_vis = tensor_to_img(vis_batch, is_grayscale=False)
img_enhanced = tensor_to_img(enhanced_vis, is_grayscale=False)
img_ir = tensor_to_img(ir_batch, is_grayscale=True)

img_fused = tensor_to_img(fused_rgb, is_grayscale=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(img_vis)
axes[0, 0].set_title('1. Low Light VIS')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_enhanced)
axes[0, 1].set_title('2. Enhanced VIS')
axes[0, 1].axis('off')

axes[1, 0].imshow(img_ir, cmap='gray')
axes[1, 0].set_title('3. Infrared (IR)')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_fused)
axes[1, 1].set_title('4. Fused')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Evaluation for Metrics

Test via test set and save results

In [ ]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
torch.cuda.empty_cache()
vis_dir = '/content/ImageFusionProject/LLVIP/visible/test'
ir_dir = '/content/ImageFusionProject/LLVIP/infrared/test'
fused_dir = '/content/fused_results'

os.makedirs(fused_dir, exist_ok=True)

import matplotlib.pyplot as plt
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

full_model = EndToEndFusionNetwork(enhancement_module).to(device)
model_path = "/content/checkpoints/colorcheckpoint/fusion_modelcolorv3_epoch_105.pth"
checkpoint = torch.load(model_path, map_location=device)

try:
    full_model.load_state_dict(checkpoint)
    print("✅ Ağırlıklar modele doğrudan başarıyla yüklendi!")
except Exception:
    full_model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Ağırlıklar sözlük (dictionary) üzerinden başarıyla yüklendi!")

full_model.eval()

test_images = [f for f in os.listdir(vis_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
print(f"Toplam {len(test_images)} görüntü 'Yama-Tabanlı (Patch-Based)' yöntemle işlenecek...\n")

with torch.no_grad():
    for filename in tqdm(test_images, desc="Füzyon İşlemi"):
        vis_path = os.path.join(vis_dir, filename)
        ir_path = os.path.join(ir_dir, filename)

        if not os.path.exists(ir_path):
            continue

        vis_img = cv2.imread(vis_path)
        vis_img = cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)
        ir_img = cv2.imread(ir_path, cv2.IMREAD_GRAYSCALE)

        h, w = vis_img.shape[:2]
        h_mid, w_mid = h // 2, w // 2

        vis_tensor = torch.from_numpy(vis_img).permute(2, 0, 1).float() / 255.0
        ir_tensor = torch.from_numpy(ir_img).unsqueeze(0).float() / 255.0

        vis_tl = vis_tensor[:, :h_mid, :w_mid].unsqueeze(0).to(device)
        vis_tr = vis_tensor[:, :h_mid, w_mid:].unsqueeze(0).to(device)
        vis_bl = vis_tensor[:, h_mid:, :w_mid].unsqueeze(0).to(device)
        vis_br = vis_tensor[:, h_mid:, w_mid:].unsqueeze(0).to(device)

        ir_tl = ir_tensor[:, :h_mid, :w_mid].unsqueeze(0).to(device)
        ir_tr = ir_tensor[:, :h_mid, w_mid:].unsqueeze(0).to(device)
        ir_bl = ir_tensor[:, h_mid:, :w_mid].unsqueeze(0).to(device)
        ir_br = ir_tensor[:, h_mid:, w_mid:].unsqueeze(0).to(device)

        fused_tl, _, _ = full_model(vis_tl, ir_tl)
        fused_tr, _, _ = full_model(vis_tr, ir_tr)
        fused_bl, _, _ = full_model(vis_bl, ir_bl)
        fused_br, _, _ = full_model(vis_br, ir_br)

        top_half = torch.cat((fused_tl, fused_tr), dim=3)
        bottom_half = torch.cat((fused_bl, fused_br), dim=3)
        fused_rgb = torch.cat((top_half, bottom_half), dim=2)

        fused_out = fused_rgb.squeeze(0).cpu().permute(1, 2, 0).numpy()
        fused_out = np.clip(fused_out * 255.0, 0, 255).astype(np.uint8)

        fused_out_bgr = cv2.cvtColor(fused_out, cv2.COLOR_RGB2BGR)
        save_path = os.path.join(fused_dir, filename)
        cv2.imwrite(save_path, fused_out_bgr)

print("\nTest seti kaydedildi.")

# Metric Calculation

In [ ]:
!pip install sewar scikit-image

Metric For LLVIP dataset

In [ ]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
torch.cuda.empty_cache()
from skimage.measure import shannon_entropy
from sewar.full_ref import vifp

vis_dir = '/content/ImageFusionProject/LLVIP/visible/test'
ir_dir = '/content/ImageFusionProject/LLVIP/infrared/test'
fused_dir = '/content/fused_results'

def calculate_sf(image):
    image = image.astype(np.float64)
    row_diff = np.diff(image, axis=0)
    col_diff = np.diff(image, axis=1)
    rf = np.sqrt(np.mean(row_diff ** 2))
    cf = np.sqrt(np.mean(col_diff ** 2))
    return np.sqrt(rf ** 2 + cf ** 2)

def calculate_scd(img_vis, img_ir, img_fused):
    img_vis = img_vis.astype(np.float64)
    img_ir = img_ir.astype(np.float64)
    img_fused = img_fused.astype(np.float64)

    diff_vis = img_fused - img_vis
    diff_ir = img_fused - img_ir

    def corr2(a, b):
        a_m, b_m = a - a.mean(), b - b.mean()
        r_num = np.sum(a_m * b_m)
        r_den = np.sqrt(np.sum(a_m ** 2) * np.sum(b_m ** 2))
        return r_num / r_den if r_den != 0 else 0

    return corr2(diff_vis, img_vis) + corr2(diff_ir, img_ir)

metrics = {"EN": [], "SD": [], "SF": [], "VIF": [], "SCD": []}

valid_images = [f for f in os.listdir(fused_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
print(f"Toplam {len(valid_images)} görüntü değerlendiriliyor. Lütfen bekleyin...")

for filename in valid_images:
    fused_path = os.path.join(fused_dir, filename)
    vis_path = os.path.join(vis_dir, filename)
    ir_path = os.path.join(ir_dir, filename)

    if not os.path.exists(vis_path) or not os.path.exists(ir_path):
        print(f"Uyarı: {filename} için orijinal IR veya VIS bulunamadı, atlanıyor.")
        continue

    img_fused = cv2.imread(fused_path, cv2.IMREAD_GRAYSCALE)
    img_vis = cv2.imread(vis_path, cv2.IMREAD_GRAYSCALE)
    img_ir = cv2.imread(ir_path, cv2.IMREAD_GRAYSCALE)

    metrics["EN"].append(shannon_entropy(img_fused))

    metrics["SD"].append(np.std(img_fused))

    metrics["SF"].append(calculate_sf(img_fused))

    vif_vis = vifp(img_vis, img_fused)
    vif_ir = vifp(img_ir, img_fused)
    metrics["VIF"].append((vif_vis + vif_ir) / 2.0)

    metrics["SCD"].append(calculate_scd(img_vis, img_ir, img_fused))

print("\n" + "="*45)
print("  VERİ SETİ ORTALAMA METRİK SONUÇLARI")
print("="*45)
for metric, values in metrics.items():
    if len(values) > 0:
        print(f"{metric}: {np.mean(values):.4f}")
    else:
        print(f"{metric}: Hesaplanamadı (Geçerli veri yok)")
print("="*45)